In [0]:
import re
import pandas as pd
from datetime import datetime, timezone

CATALOG = "estudo_estrangeiros"
VOLUME_PATH = f"/Volumes/estudo_estrangeiros/bronze/input_dados_fbref"
SEASONS = list(range(2014, 2027))  # 2014-2025 sendo as temporadas completas

TABLES = [
    "player_standard_stats",
    "player_playing_time",
    "squad_playing_time",
    "nationalities",
    "league_table",
]

def find_header_row(path):
    with open(path, encoding="utf-8-sig") as f:
        for i, line in enumerate(f):
            fields = line.rstrip("\n").split(",")
            non_empty = sum(1 for x in fields if x.strip() != "")
            if len(fields) >= 4 and non_empty == len(fields):
                return i
    raise ValueError(f"Não encontrei linha de cabeçalho em {path}")

def sanitize_columns(pdf):
    pdf.columns = [re.sub(r"[ ,;{}()\n\t=]+", "_", str(c)).strip("_") for c in pdf.columns]
    return pdf

def load_bronze_table(table_name, season):
    raw_path = f"{VOLUME_PATH}/{table_name}/{season}.csv"
    header_row = find_header_row(raw_path)
    if header_row is None:
        print(f"  [aviso] {raw_path} vazio - pulando {season}")
        return None

    pdf = pd.read_csv(raw_path, skiprows=header_row, encoding="utf-8-sig")
    pdf = sanitize_columns(pdf)
    pdf = pdf[~pdf.apply(lambda row: row.astype(str).str.contains("Provided by", case=False, na=False).any(), axis=1)]

    if pdf.empty:
        print(f"  [aviso] {raw_path} sem linhas de dados ainda - pulando {season}")
        return None
    
    # Bronze mantém tudo como texto bruto - formato muda entre temporadas
    # (ex: "Age" vem como "25" em temporada encerrada, "27-166" na temporada em andamento)
    for col in pdf.columns:
        pdf[col] = pdf[col].apply(lambda x: str(x) if pd.notnull(x) else None)
    

    pdf["_season"] = season
    pdf["_source_table"] = table_name
    pdf["_source_file"] = raw_path
    pdf["_ingested_at"] = datetime.now(timezone.utc).isoformat()
    return pdf

for table in TABLES:
    season_frames = [load_bronze_table(table, season) for season in SEASONS]
    full_pdf = pd.concat(season_frames, ignore_index=True)
    df = spark.createDataFrame(full_pdf)

    target = f"{CATALOG}.bronze.{table}"
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(target)
    print(f"{target}: {df.count()} linhas, {len(df.columns)} colunas ({len(SEASONS)} temporadas)")


estudo_estrangeiros.bronze.player_standard_stats: 9620 linhas, 30 colunas (13 temporadas)
estudo_estrangeiros.bronze.player_playing_time: 11376 linhas, 30 colunas (13 temporadas)
estudo_estrangeiros.bronze.squad_playing_time: 260 linhas, 23 colunas (13 temporadas)
estudo_estrangeiros.bronze.nationalities: 250 linhas, 9 colunas (13 temporadas)
estudo_estrangeiros.bronze.league_table: 260 linhas, 20 colunas (13 temporadas)
